In [ ]:
# ================================================================================
# PhyloCNN training under the BDEIVH model
# ================================================================================
#
# This notebook trains a convolutional neural network (PhyloCNN) to infer
# epidemiological parameters from encoded phylogenetic trees generated under
# the Birth–Death Epidemiological model with exposed-infectious Vectors
# and Hosts (BDEIVH).
#
# The implementation was adapted from the original PhyloCNN framework:
#
#     https://github.com/manolofperez/phyloCNN
#
# ================================================================================

# Load the libraries required for data manipulation, model construction,
# training, and serialization of custom Keras objects.

import pandas as pd
import numpy as np
import tensorflow as tf
import keras

from keras.models import Sequential, Model
from keras.saving import register_keras_serializable
from keras.layers import (
    Activation,
    Dense,
    Conv2D,
    GlobalAveragePooling2D,
    BatchNormalization,
    Dropout,
    Flatten
)

# Quantiles predicted by the model:
# lower bound (2.5%), median (50%), and upper bound (97.5%).

QUANTILES = (0.025, # lower
             0.5,   # median
             0.975) # upper

In [ ]:
# Custom pinball loss function for simultaneous quantile regression.
# The model predicts three quantiles (2.5%, 50%, and 97.5OV%) for each target
# parameter. This loss penalizes under- and over-estimation asymmetrically,
# according to the specified quantiles.

# Input shapes:
#
# y_true:
#     (batch_size, n_targets)
#
# y_pred (raw network output):
#     (batch_size, n_targets × n_quantiles)
#
# Example for six target parameters:
#
# y_true.shape = (batch_size, 6)
# y_pred.shape = (batch_size, 18)
#
# After reshaping:
#
# y_pred.shape = (batch_size, 6, 3)
#
# where the last dimension corresponds to:
# [lower, median, upper]

@register_keras_serializable(package="bdeissct_dl", name="pinball_loss")
def pinball_loss(y_true, y_pred):
    # Convert the tuple of quantiles into a TensorFlow constant.
    quantiles_tensor = tf.constant(QUANTILES, dtype=tf.float32)

    # Number of target parameters predicted by the model.
    n_targets = tf.shape(y_true)[-1]

    # Number of quantiles predicted for each target.
    n_quantiles = len(QUANTILES)

    # Reshape predictions from (batch, n_targets × n_quantiles)
    # to (batch, n_targets, n_quantiles).
    y_pred = tf.reshape(y_pred, (-1, n_targets, n_quantiles))

    # Add a singleton dimension so that each true value can be compared
    # simultaneously with its three predicted quantiles through
    # TensorFlow broadcasting.
    y_true = tf.expand_dims(y_true, axis=-1)

    # Compute prediction errors.
    #
    # Positive error:
    #     prediction < true value
    #
    # Negative error:
    #     prediction > true value
    errors = y_true - y_pred

    # Compute the pinball loss for each quantile.

    # Pinball (quantile) loss:
    #
    # rho_q(e) = max(qe, (q-1)e)
    #
    # where:
    #
    #     e = y_true - y_pred
    #     q = quantile level
    #
    # The asymmetric penalty encourages each output neuron to estimate
    # the specified conditional quantile.
    loss_per_quantile = tf.maximum(
        quantiles_tensor * errors,
        (quantiles_tensor - 1.0) * errors
    )

    # Return the average loss across all targets, quantiles, and samples.
    return tf.reduce_mean(loss_per_quantile)

In [ ]:
# Load the parameters and encodings used to train the model.

# Tree parameters for the training set.
param_train = pd.read_csv(
    './datasets/250_350_train_merged_all_OROV.log', sep=','
)

# Tree encodings for the training set.
encoding = pd.read_csv(
    './datasets/250_350_train_merged_all',
    sep="\t",
    header=0,
    index_col=0
).values.astype('float32').reshape(-1, 800, 11)

# Match parameter rows to encoding rows.
param_train = param_train.iloc[:encoding.shape[0]].reset_index(drop=True)

print(f"Train: {encoding.shape[0]} trees | {param_train.shape[0]} parameter rows")
print(param_train.shape)


In [ ]:
# Define targets and prepare the training labels.

target_1 = "Rvh"        # Transmission from vectors to humans
target_2 = "Rhv"        # Transmission from humans to vectors
target_3 = "Rhh"        # Transmission among humans
target_4 = "d_v"        # Total duration of the infection in vectors
target_5 = "d_h"        # Total duration of the infection in humans
target_6 = "v2h_ratio"  # Ratio of vectors and humans

target_cols = [target_1, target_2, target_3, target_4, target_5, target_6]

# Extract the rescaling factor from the last row of every encoded tree.
param_train['resc_factor'] = encoding[:, -1, -1]

# Rescale time-dependent targets.
for key in (target_4, target_5):
    param_train[key] = param_train[key] / param_train['resc_factor']

# Remove the final row containing the rescaling factor.
encoding = np.delete(encoding, -1, axis=1)

# Labels for each simulation.
Y = pd.DataFrame(param_train[target_cols]).reset_index(drop=True)

# Fraction used internally for validation.
valid_frac = 0.3

print("Y train:", Y.shape)


In [ ]:
# Add the tree-level sampling probability (p) as an additional feature
# repeated across every encoded row.

n_rows = encoding.shape[1]
samp_proba_list = np.asarray(param_train['p'])

encoding = np.concatenate(
    (encoding, np.repeat(samp_proba_list, n_rows).reshape(-1, n_rows, 1)),
    axis=2
)

print("Training encoding:", encoding.shape)


In [ ]:
# Transform unconstrained network outputs into positive,
# ordered, and non-crossing quantile predictions.
#
# For each target parameter, the network initially produces:
#
#     raw_lower
#     raw_gap_to_median
#     raw_gap_to_upper
#
# The final outputs are constructed as:
#
#     lower  = softplus(raw_lower)
#     median = lower + softplus(raw_gap_to_median)
#     upper  = median + softplus(raw_gap_to_upper)
#
# Therefore:
#
#     0 <= lower <= median <= upper
#
# The final outputs continue to represent the predicted
# 2.5%, 50%, and 97.5% quantiles.

@register_keras_serializable(
    package="bdeissct_dl",
    name="ordered_positive_quantiles"
)
def ordered_positive_quantiles(raw, n_targets=6):

    # Reshape:
    #
    # (batch_size, 18)
    # ->
    # (batch_size, 6, 3)
    raw = tf.reshape(
        raw,
        (-1, n_targets, len(QUANTILES))
    )

    # Extract the three unrestricted intermediate values.
    raw_lower = raw[..., 0]
    raw_gap_to_median = raw[..., 1]
    raw_gap_to_upper = raw[..., 2]

    # Transform the first unrestricted output into a non-negative
    # lower quantile using the Softplus function.
    lower = tf.nn.softplus(raw_lower)

    # Transform the second unrestricted output into a non-negative
    # increment from the lower quantile to the median.
    median = lower + tf.nn.softplus(
        raw_gap_to_median
    )

    # Non-negative increment from median to upper.
    upper = median + tf.nn.softplus(
        raw_gap_to_upper
    )

    # Arrange the final predictions in the order expected
    # by the pinball loss: [lower, median, upper].
    quantiles = tf.stack(
        [lower, median, upper],
        axis=-1
    )

    # Return:
    #
    # (batch_size, 6, 3)
    # ->
    # (batch_size, 18)
    return tf.reshape(
        quantiles,
        (-1, n_targets * len(QUANTILES))
    )

In [ ]:
# Define and return the PhyloCNN architecture used to predict
# the epidemiological target parameters and their quantiles.

def build_model():
    # Initialize the Sequential model
    model = Sequential()

    # First convolutional layer:
    # - Filters: 32
    # - Kernel size: (1, 12), sliding across the second dimension of the input
    # - Input shape: (400, 12, 2) where 400 is the number of tree leaves/nodes, 12 is the feature size, and 2 is the number of channels (leaves and nodes)
    # - Activation function: ELU (Exponential Linear Unit)
    # - Groups: 2 to apply separate convolutions for the two channels (leaves and nodes)
    model.add(Conv2D(filters=32, use_bias=False, kernel_size=(1, 12), input_shape=(400, 12, 2), activation='elu', groups=2))

    # Apply batch normalization to stabilize and speed up the training process
    model.add(BatchNormalization())

    # Second convolutional layer:
    # - Filters: 32
    # - Kernel size: (1, 1) to process each feature independently
    # - Activation function: ELU
    model.add(Conv2D(filters=32, use_bias=False, kernel_size=(1, 1), activation='elu'))

    # Apply batch normalization again
    model.add(BatchNormalization())

    # Third convolutional layer:
    # - Filters: 32
    # - Kernel size: (1, 1) for further feature processing
    # - Activation function: ELU
    model.add(Conv2D(filters=32, use_bias=False, kernel_size=(1, 1), activation='elu'))

    # Apply batch normalization for the final time before flattening
    model.add(BatchNormalization())

    # Flatten the 2D feature maps from the convolutional layers into a 1D vector,
    # which will be passed to the fully connected (dense) layers
    model.add(GlobalAveragePooling2D())

    # Fully connected (FFNN) part:
    # Dense layers with decreasing number of units, all using ELU activation:
    model.add(Dense(64, activation='elu'))  # First dense layer with 64 units
    model.add(Dense(32, activation='elu'))  # Second dense layer with 32 units
    model.add(Dense(16, activation='elu'))  # Third dense layer with 16 units
    model.add(Dense(8, activation='elu'))  # Fourth dense layer with 8 units

    # Current output layer:
    # - Produces 18 values: 6 target parameters × 3 quantiles.
    # - The outputs are ordered as [lower, median, upper] for each target.
    # - Linear activation leaves every output unconstrained.
    # - Consequently, the three quantiles are predicted independently and
    #   may become negative or cross one another.

    ### ATENTION: THIS WAS THE ORIGINAL LINE THAT WAS CHANGED TO
    ### CONSTRAIN THE PREDICTED VALUES
    # model.add(Dense(6 * len(QUANTILES), activation='linear'))

    # Produce 18 unrestricted intermediate values:
    # six target parameters × three values per parameter.
    model.add(
        Dense(
            6 * len(QUANTILES),
            activation="linear",
            name="raw_quantile_outputs"
        )
    )

    # Transform the intermediate values into positive,
    # ordered quantile predictions.
    model.add(
        keras.layers.Lambda(
            ordered_positive_quantiles,
            arguments={"n_targets": 6},
            output_shape=(6 * len(QUANTILES),),
            name="ordered_positive_quantiles"
        )
    )

    # Show the model architecture, output shapes, and number of parameters.
    model.summary()

    # Return the constructed Keras model.
    return model

In [ ]:
# This function converts each encoded tree into two fixed-length channels:
# one for leaves and one for internal nodes.
#
# Leaves and internal nodes are sorted by distance from the root and
# zero-padded to a maximum of 400 rows per channel.

def encode_pad_0s_rootage_single(enc, max_len=400):
    enc_pad = []

    for i in range(enc.shape[0]):
        # Leaves: column 3 == 1.
        leaves = enc[i][enc[i, :, 3] == 1]
        leaves = leaves[np.argsort(leaves[:, 1])]
        leaves = np.pad(leaves, [(0, (max_len - leaves.shape[0])), (0, 0)], mode='constant')

        # Internal nodes: column 3 > 1.
        nodes = enc[i][enc[i, :, 3] > 1]
        nodes = nodes[np.argsort(nodes[:, 1])]

        # Copy the last node's value to balance leaves and nodes.
        # This follows the original notebook behavior.
        nodes = np.append(nodes, nodes[-1].reshape(1, -1), axis=0)
        nodes = np.pad(nodes, [(0, (max_len - nodes.shape[0])), (0, 0)], mode='constant')

        enc_pad.append(np.stack((leaves, nodes), axis=2))

    return np.array(enc_pad)

# Change encoding to order by root age and pad with 0s.
encoding_pad = encode_pad_0s_rootage_single(encoding)

# Verify compatibility with the model input shape: (400, 12, 2).
print("Training encoding:", encoding_pad.shape)

In [ ]:
# Plot the training and validation loss curves produced during model
# training. Optionally highlight the best epoch and save the figure
# as a PDF.

import matplotlib.pyplot as plt

def plot_losses(history, pdf=None, best_epoch=None, skip_first=0):

    # Retrieve the training and validation loss recorded at each epoch.
    train_loss = history.history.get("loss", [])
    val_loss = history.history.get("val_loss", [])

    if not train_loss:
        # Stop if the History object does not contain training losses.
        raise ValueError("No 'loss' found in history.history")

    # Ignore the requested number of initial epochs.
    start = max(0, skip_first)

    # Epoch numbers shown on the x-axis.
    epochs = range(start + 1, len(train_loss) + 1)

    # Create the figure.
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_loss[start:], label="Training loss")

    # Plot the validation loss if available.
    if val_loss:
        plt.plot(epochs, val_loss[start:], label="Validation loss")

    # Highlight the epoch selected as the best model.
    if best_epoch is not None:
        # Draw a vertical line indicating the selected epoch.
        plt.axvline(best_epoch + 1, linestyle="--", alpha=0.7)
        if val_loss and best_epoch < len(val_loss):
            # Mark the validation loss at the selected epoch.
            plt.scatter(best_epoch + 1, val_loss[best_epoch], s=100, marker="*")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and validation loss")
    plt.legend()
    plt.tight_layout()

    # Save the figure if an output filename was provided.
    if pdf is not None:
        plt.savefig(pdf, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
# Create the output directory used to store the trained model.
# No error is raised if the directory already exists.
import os
os.makedirs('./trained_models', exist_ok=True)

# Initialize the model using the build_model function that was previously defined
estimator = build_model()

# Compile the model:
#
# - loss:
#     Custom pinball loss used to train the 2.5%, 50%, and 97.5%
#     conditional quantiles simultaneously.
#
# - optimizer:
#     Adam updates the network weights using gradients of the
#     pinball loss.
#
# No additional evaluation metrics are specified.
estimator.compile(loss=pinball_loss,optimizer='Adam')

# Early stopping callback to prevent overfitting:
# - monitor: monitor the validation loss during training
# - patience: stop training if the validation loss doesn't improve for 15 consecutive epochs
# - mode: 'min' indicates that training will stop when the validation loss reaches its minimum
# - restore_best_weights: restore the weights from the best epoch (the one with the lowest validation loss)
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, mode='min', restore_best_weights=True, min_delta=1e-3)

# Custom callback to display training progress:
# - Print a dot for every epoch (or newline every 100 epochs) to indicate progress in training
class PrintD(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs):
        if epoch % 100 == 0:  # Print a newline every 100 epochs
            print('')
        print('.', end='')  # Print a dot to indicate progress during each epoch

# Maximum number of training epochs. Early stopping may terminate
# training before this limit is reached.
EPOCHS = 1000

# Verify that the prepared training data are compatible with
# the expected model input shape.
print("encoding shape:", encoding.shape)
print("encoding_pad shape:", encoding_pad.shape)
print("model input shape:", estimator.input_shape)

# Train the model using the `fit` method:
# - encoding_pad: The padded training data (inputs)
# - Y: The target values (outputs)
# - verbose: set to 1 to print progress during training
# - epochs: The number of times to iterate over the entire dataset
# - validation_split: the fraction of data to use for validation (used to monitor validation loss)
# - batch_size: the number of samples per gradient update
# - callbacks: list of callbacks to be used during training (early stopping and progress display)
history = estimator.fit(encoding_pad, Y, verbose=1, epochs=EPOCHS, validation_split=valid_frac, batch_size=1024, callbacks=[early_stop, PrintD()])

# Save the model architecture to a JSON file:
# - The model structure (architecture) is saved as a JSON string
from keras.models import model_from_json
model = estimator.to_json()
with open('./trained_models/250_350_OROV_OROV_trained_BDEIVH.json', 'w') as json_file:
    json_file.write(model)

# Save the model weights to an H5 file:
# - The weights (learned parameters) of the trained model are saved to a file
estimator.save_weights('./trained_models/250_350_OROV_OROV_trained_BDEIVH.weights.h5')

# Print a confirmation message when the model and weights are saved
print("Model architecture and weights saved successfully.")


In [ ]:
# Determine the epoch corresponding to the best model.
#
# If EarlyStopping interrupted training, recover the epoch with the
# lowest validation loss. Otherwise, use the final training epoch.
if early_stop.stopped_epoch == 0:
    best_epoch = len(history.history["loss"]) - 1
else:
    best_epoch = (
        early_stop.stopped_epoch
        - early_stop.patience
    )

# Plot the training and validation loss curves and save the figure.
plot_losses(
    history,
    pdf="./trained_models/250_350_OROV_OROV_trained_BDEIVH_losses.pdf",
    best_epoch=best_epoch
)